In [ ]:
import os
from google.colab import drive
import pandas as pd
import uuid
import json
import numpy as np

In [ ]:
drive.mount('/content/drive')

#Data Loading

In [ ]:
data_directory = "/content/drive/MyDrive/Colab Notebooks/MastersResearchProject/dataset/"
filenames = os.listdir(data_directory)
post_filenames = sorted([f for f in filenames if f.endswith("post.csv")])
comment_filenames = sorted([f for f in filenames if f.endswith("comment.csv")])
print(filenames)
print(post_filenames)
print(comment_filenames)

In [ ]:
posts_all = [pd.read_csv(data_directory + f) for f in post_filenames]
comments_all = [pd.read_csv(data_directory + f) for f in comment_filenames]

In [ ]:
sum_post = 0
sum_comment = 0
for i, (post_df, comment_df) in enumerate(zip(posts_all, comments_all)):
    print(f"{len(post_df):6} {len(comment_df):6} {post_filenames[i]}")
    sum_post += len(post_df)
    sum_comment += len(comment_df)
print(f"{sum_post} {sum_comment}")

In [ ]:
all_posts_authors = pd.concat([post_df["Post_Author"] for post_df in posts_all])
all_comments_authors = pd.concat([comment_df["Comment_Author"] for comment_df in comments_all])
all_authors = pd.concat([all_posts_authors, all_comments_authors]).unique()
author_map = {author: str(uuid.uuid4()) for author in all_authors}

In [ ]:
print(f"{len(all_posts_authors):6} {len(all_comments_authors):6}")
print(f"{len(author_map)} authors")
print(author_map.keys())

In [ ]:

reversed_author_map = {value: key for key, value in author_map.items()}
print(reversed_author_map)

#Data Cleaning

In [ ]:
pip install emoji

In [ ]:
import re
import emoji

def _emoji_to_words(emj, data):
    name = data.get("en", "").strip(":").replace("_", " ").replace("-", " ")
    return f" {name} " # pad so it never glues to adjacent text

def demojize_text(text):
    if not isinstance(text, str):
        return text
    out = emoji.replace_emoji(text, replace=_emoji_to_words)
    out = re.sub(r"[^\S\n]+", " ", out) # collapse horizontal whitespace, keep newlines
    out = re.sub(r" *\n *", "\n", out)
    return out.strip()

In [ ]:
posts_all_clean = []
comments_all_clean = []
for post_df, comments_df in zip(posts_all, comments_all):
    posts_clean = post_df[["Post_ID","Post_Content","Post_Author","Post_Date","Post_Time"]].copy()
    posts_clean["Author_ID"] = posts_clean["Post_Author"].map(author_map)
    posts_clean['Post_Date'] = posts_clean['Post_Date'].str.replace(u'\u200e', '', regex=True)
    posts_clean['Post_Time'] = posts_clean['Post_Time'].str.replace(u'\u200e', '', regex=True)
    posts_clean["Timestamp"] = pd.to_datetime(posts_clean["Post_Date"] + " " + posts_clean["Post_Time"], format='%d-%m-%Y %I:%M %p', errors="coerce")
    posts_clean = posts_clean.drop(columns=["Post_Author", "Post_Date","Post_Time"])

    posts_clean = posts_clean.dropna(subset=["Post_Content","Timestamp"])
    posts_clean = posts_clean.drop_duplicates(subset=["Post_Content", "Timestamp", "Author_ID"])

    # Translate Emoji
    posts_clean["Post_Content"]    = posts_clean["Post_Content"].apply(demojize_text)

    posts_clean = posts_clean.sort_values(["Post_ID"])
    posts_all_clean.append(posts_clean)

    comments_clean = comments_df[["Post_ID","Comment_ID","Comment_Content","Comment_Author","Comment_Date","Comment_Time"]].copy()
    comments_clean["Author_ID"] = comments_clean["Comment_Author"].map(author_map).fillna("00000000-0000-0000-0000-000000000000")
    comments_clean['Comment_Date'] = comments_clean['Comment_Date'].str.replace(u'\u200e', '', regex=True)
    comments_clean['Comment_Time'] = comments_clean['Comment_Time'].str.replace(u'\u200e', '', regex=True)
    comments_clean["Timestamp"] = pd.to_datetime(comments_clean["Comment_Date"] + " " + comments_clean["Comment_Time"], format='%d-%m-%Y %I:%M %p', errors="coerce")
    comments_clean = comments_clean.drop(columns=["Comment_Author", "Comment_Date","Comment_Time"])

    comments_clean = comments_clean.dropna(subset=["Comment_Content","Timestamp"])
    comments_clean = comments_clean.drop_duplicates(subset=["Comment_Content", "Timestamp", "Author_ID"])

    # Translate Emoji
    comments_clean["Comment_Content"] = comments_clean["Comment_Content"].apply(demojize_text)

    # Remove comments with no post
    comments_clean = comments_clean[comments_clean["Post_ID"].isin(posts_clean["Post_ID"])]

    # Flag comments made by the post's author
    post_author_map = posts_clean.drop_duplicates("Post_ID").set_index("Post_ID")["Author_ID"]
    comments_clean["Is_Poster"] = (comments_clean["Author_ID"] == comments_clean["Post_ID"].map(post_author_map))

    comments_clean = comments_clean.sort_values(["Post_ID", "Comment_ID"])
    comments_all_clean.append(comments_clean)

In [ ]:
comments_all_clean[1].head(20)

In [ ]:
sum_post = 0
sum_comment = 0
for i, (post_df, comment_df) in enumerate(zip(posts_all_clean, comments_all_clean)):
    print(f"{len(post_df):6} {len(comment_df):6} {post_filenames[i]}")
    sum_post += len(post_df)
    sum_comment += len(comment_df)
print(f"{sum_post} {sum_comment}")

In [ ]:
"""
Remove usernames that have already posted in thread in comments
"""

MASK = "someone"


def _mask_text(text, escaped_names):
    if not isinstance(text, str) or not escaped_names:
        return text
    pattern = re.compile(r"\b(?:" + "|".join(escaped_names) + r")\b", flags=re.IGNORECASE)
    return pattern.sub(MASK, text)


def apply_running_mask(posts_clean, comments_clean, author_col="Author_ID"):
    posts_clean = posts_clean.copy()
    comments_clean = comments_clean.copy()

    comments_by_post = {pid: g for pid, g in comments_clean.groupby("Post_ID", sort=False)}
    post_out, comment_out = {}, {}

    for pidx, prow in posts_clean.iterrows():
        running, seen = [], set()

        def add(uid):
            name = reversed_author_map.get(uid)
            if isinstance(name, str):
                key = name.strip().lower()
                if key and key not in seen:
                    seen.add(key)
                    running.append(re.escape(name.strip()))

        def ordered():  # longest first so overlapping names don't partial-match
            return sorted(running, key=len, reverse=True)

        add(prow[author_col]) # post author has occurred
        post_out[pidx] = _mask_text(prow["Post_Content"], ordered())

        g = comments_by_post.get(prow["Post_ID"])
        if g is not None:
            for cidx, crow in g.sort_values("Comment_ID").iterrows():
                add(crow[author_col])                    # this author now occurs
                comment_out[cidx] = _mask_text(crow["Comment_Content"], ordered())

    posts_clean["Post_Content"] = [post_out.get(i, c)
        for i, c in zip(posts_clean.index, posts_clean["Post_Content"])]
    comments_clean["Comment_Content"] = [comment_out.get(i, c)
        for i, c in zip(comments_clean.index, comments_clean["Comment_Content"])]
    return posts_clean, comments_clean

In [ ]:
for i, (post_df, comments_df) in enumerate(zip(posts_all_clean, comments_all_clean)):
    posts_all_clean[i], comments_all_clean[i] = apply_running_mask(post_df, comments_df)

In [ ]:
sum_post = 0
sum_comment = 0
for i, (post_df, comment_df) in enumerate(zip(posts_all_clean, comments_all_clean)):
    print(f"{len(post_df):6} {len(comment_df):6} {post_filenames[i]}")
    sum_post += len(post_df)
    sum_comment += len(comment_df)
print(f"{sum_post} {sum_comment}")

In [ ]:
posts_all_clean[1].head(10)

In [ ]:
comments_all_clean[1].head(10)

In [ ]:
save_location = "/content/drive/MyDrive/Colab Notebooks/MastersResearchProject/dataset_clean/"

with open(save_location + "author_map.json", "w") as f:
    json.dump(author_map, f)

for filename, post in zip(post_filenames, posts_all_clean):
    print(f"{save_location} {filename[:-4]} + _cleaned.csv {len(post)}")
    post.to_csv(save_location + filename[:-4] + "_cleaned.csv", index=False)

for filename, comment in zip(comment_filenames, comments_all_clean):
    print(f"{save_location} {filename[:-4]} + _cleaned.csv {len(comment)}")
    comment.to_csv(save_location + filename[:-4] + "_cleaned.csv", index=False)

#Data Processing into Emotions



In [ ]:
import re
import ast
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os, json, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

In [ ]:
MODEL_NAME = "borisn70/bert-43-multilabel-emotion-detection"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

sbert_model = SentenceTransformer("all-mpnet-base-v2") # 768-d

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_sentences(text: str) -> list[str]:
    # Protect common abbreviations
    abbrevs = r"\b(Mr|Mrs|Ms|Dr|Prof|Sr|Jr|vs|etc|e\.g|i\.e)\."
    text = re.sub(abbrevs, r"\1<DOT>", text, flags=re.IGNORECASE)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    # Restore protected dots and filter empty strings
    sentences = [s.replace("<DOT>", ".").strip() for s in sentences if s.strip()]
    return sentences

def predict_emotions(sentences: list[str], tokenizer, model, batch_size: int = 32, device: torch.device = torch.device("cpu"),) -> list[list[float]]:
    all_vectors = []
    model.to(device)

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            logits = model(**encoded).logits
            probs  = torch.sigmoid(logits).cpu()

        all_vectors.extend(probs.tolist())

    return all_vectors # list of list[float]

def forum_key_from_filename(filename):
    key = filename.replace(".csv", "").replace("_cleaned", "")
    return key.replace("_post", "").replace("_comment", "")

def make_uid(forum_key, post_id):
    return f"{forum_key}::{post_id}"


In [ ]:
TEST_FRACTION = 0.25
RANDOM_SEED   = 0
CUTOFF_FRACTIONS = [0.25, 0.50, 0.75]

input_directory  = "/content/drive/MyDrive/Colab Notebooks/MastersResearchProject/dataset_clean/"
feature_directory = "/content/drive/MyDrive/Colab Notebooks/MastersResearchProject/dataset_features/"

In [ ]:
filenames = os.listdir(input_directory)
post_filenames = sorted([f for f in filenames if f.endswith("post_cleaned.csv")])
comment_filenames = sorted([f for f in filenames if f.endswith("comment_cleaned.csv")])
print(filenames)
print(post_filenames)
print(comment_filenames)

In [ ]:
forum_files = {}
for filename in sorted(os.listdir(input_directory)):
    if not filename.endswith(".csv"):
        continue
    forum_key = forum_key_from_filename(filename)
    forum_files.setdefault(forum_key, {})
    if "post" in filename:
        forum_files[forum_key]["post"] = filename
    elif "comment" in filename:
        forum_files[forum_key]["comment"] = filename

# 25% of threads per forum
random_generator = np.random.default_rng(RANDOM_SEED)
uid_to_split = {}
uid_to_forum = {}
for forum_key, files in forum_files.items():
    if "post" not in files:
        continue
    forum_posts = pd.read_csv(input_directory + files["post"])
    unique_post_ids = forum_posts["Post_ID"].unique()
    shuffled_post_ids = random_generator.permutation(unique_post_ids)
    number_of_test_posts = int(round(TEST_FRACTION * len(shuffled_post_ids)))
    test_post_ids = set(shuffled_post_ids[:number_of_test_posts])
    for post_id in unique_post_ids:
        uid = make_uid(forum_key, post_id)
        uid_to_split[uid] = "test" if post_id in test_post_ids else "train"
        uid_to_forum[uid] = forum_key

print(f"forums paired: {len(forum_files)} | threads assigned: {len(uid_to_split)}")

In [ ]:
comment_feature_frames = []

for forum_key, files in forum_files.items():
    if "comment" not in files:
        continue
    forum_comments = pd.read_csv(input_directory + files["comment"])
    comment_records = []

    for _, comment_row in forum_comments.iterrows():
        post_id    = comment_row["Post_ID"]
        comment_id = comment_row["Comment_ID"]

        sentences = split_sentences(clean_text(comment_row["Comment_Content"]))
        if not sentences:
            continue

        sentence_emotion_vectors = predict_emotions(sentences, tokenizer, model, device=device)
        comment_emotion_vector = np.mean(np.stack(sentence_emotion_vectors), axis=0)

        comment_records.append({
            "UID":             make_uid(forum_key, comment_row["Post_ID"]),
            "Post_ID":         comment_row["Post_ID"],
            "forum":           forum_key,
            "Comment_ID":      comment_row["Comment_ID"],
            "Author_ID":       comment_row["Author_ID"],
            "Timestamp":       comment_row["Timestamp"],
            "Is_Poster":       comment_row["Is_Poster"],
            "Comment_Text":    " ".join(sentences),
            "Emotion_Vector":  comment_emotion_vector,
        })

    forum_frame = pd.DataFrame(comment_records)
    if forum_frame.empty:
        continue

    # batch SBERT over the whole forum
    sbert_matrix = sbert_model.encode(forum_frame["Comment_Text"].tolist(),
                                      normalize_embeddings=True,
                                      batch_size=64, show_progress_bar=True)
    forum_frame["SBERT_Vector"] = list(sbert_matrix)

    forum_frame["forum"] = forum_key
    forum_frame["split"] = forum_frame["UID"].map(uid_to_split)

    comment_feature_frames.append(forum_frame)

comments_all = pd.concat(comment_feature_frames, ignore_index=True)

# save vectors as json strings
comments_to_save = comments_all.copy()
comments_to_save["Emotion_Vector"] =  ["Emotion_Vector"].apply(lambda v: json.dumps(np.asarray(v).tolist()))
comments_to_save["SBERT_Vector"]   = comments_to_save["SBERT_Vector"].apply(lambda v: json.dumps(np.asarray(v).tolist()))
comments_to_save.to_csv(feature_directory + "comments_features.csv", index=False)
print("comments_all:", comments_all.shape, "|", comments_all["split"].value_counts().to_dict())

In [ ]:
post_feature_frames = []

for forum_key, files in forum_files.items():
    if "post" not in files:
        continue
    forum_posts = pd.read_csv(input_directory + files["post"])
    post_records = []

    for _, post_row in forum_posts.iterrows():
        post_id = post_row["Post_ID"]

        sentences = split_sentences(clean_text(post_row["Post_Content"]))
        if not sentences:
            continue

        sentence_emotion_vectors = predict_emotions(sentences, tokenizer, model, device=device)
        post_emotion_vector = np.mean(np.stack(sentence_emotion_vectors), axis=0)

        post_records.append({
            "UID":            make_uid(forum_key, post_row["Post_ID"]),
            "Post_ID":        post_row["Post_ID"],
            "forum":          forum_key,
            "Author_ID":      post_row["Author_ID"],
            "Timestamp":      post_row["Timestamp"],
            "Post_Text":      " ".join(sentences),
            "Emotion_Vector": post_emotion_vector,
        })

    forum_frame = pd.DataFrame(post_records)
    if forum_frame.empty:
        continue

    sbert_matrix = sbert_model.encode(forum_frame["Post_Text"].tolist(),
                                      normalize_embeddings=True,
                                      batch_size=64, show_progress_bar=True)
    forum_frame["SBERT_Vector"] = list(sbert_matrix)
    forum_frame["forum"] = forum_key
    forum_frame["split"] = forum_frame["UID"].map(uid_to_split)

    post_feature_frames.append(forum_frame)

posts_all = pd.concat(post_feature_frames, ignore_index=True)

posts_to_save = posts_all.copy()
posts_to_save["Emotion_Vector"] = posts_to_save["Emotion_Vector"].apply(lambda v: json.dumps(np.asarray(v).tolist()))
posts_to_save["SBERT_Vector"]   = posts_to_save["SBERT_Vector"].apply(lambda v: json.dumps(np.asarray(v).tolist()))
posts_to_save.to_csv(feature_directory + "posts_features.csv", index=False)
print("posts_all:", posts_all.shape, "|", posts_all["split"].value_counts().to_dict())

In [ ]:
import pandas as pd, numpy as np, json

comments_all = pd.read_csv(feature_directory + "comments_features.csv")
posts_all    = pd.read_csv(feature_directory + "posts_features.csv")

for df, cols in [(comments_all, ["Emotion_Vector", "SBERT_Vector"]),
                 (posts_all,    ["Emotion_Vector", "SBERT_Vector"])]:
    for c in cols:
        df[c] = df[c].apply(lambda v: np.array(json.loads(v), dtype=np.float32))

In [ ]:
import pickle

def stack_mean(series_of_vectors):
    arr = np.stack([np.asarray(v, dtype=np.float32) for v in series_of_vectors])
    if np.isnan(arr).all():
        return np.full(arr.shape[1], np.nan).tolist() # no valid comment at all -> genuine NaN
    return np.nanmean(arr, axis=0).tolist() # skip NaN comments, average the rest

def as_sequence_array(series_of_vectors):
    arr = np.stack([np.asarray(v, dtype=np.float32) for v in series_of_vectors]).astype(np.float32)
    keep = ~np.isnan(arr).all(axis=1)
    return np.nan_to_num(arr[keep]) if keep.any() else np.nan_to_num(arr)

post_emotion_lookup = {row["UID"]: np.asarray(row["Emotion_Vector"], dtype=np.float32)
                       for _, row in posts_all.iterrows()}
post_sbert_lookup   = {row["UID"]: np.asarray(row["SBERT_Vector"], dtype=np.float32)
                       for _, row in posts_all.iterrows()}

for cutoff_fraction in CUTOFF_FRACTIONS:
    cutoff_percentage = int(cutoff_fraction * 100)
    thread_rows = []
    sequence_store = {}

    for uid, thread in comments_all.groupby("UID"):          # group by UID, not Post_ID
        thread = thread.sort_values(["Timestamp", "Comment_ID"])
        number_of_comments = len(thread)
        cutoff_index = int(round(cutoff_fraction * number_of_comments))
        if cutoff_index < 1 or cutoff_index >= number_of_comments:
            continue
        if uid not in post_emotion_lookup:
            continue

        context_comments = thread.iloc[:cutoff_index]
        target_comments  = thread.iloc[cutoff_index:]
        poster_target_comments = target_comments[target_comments["Is_Poster"] == True]

        post_emotion_vector = post_emotion_lookup[uid]     # (43,)
        post_sbert_vector   = post_sbert_lookup[uid]       # (768,)

        comment_emotion_sequence = as_sequence_array(context_comments["Emotion_Vector"])  # (T,43)
        comment_sbert_sequence   = as_sequence_array(context_comments["SBERT_Vector"])    # (T,768)

        emotion_with_post = np.vstack([post_emotion_vector[None, :], comment_emotion_sequence])
        sbert_with_post   = np.vstack([post_sbert_vector[None, :],   comment_sbert_sequence])
        is_post_flag      = np.zeros((emotion_with_post.shape[0], 1), dtype=np.float32)
        is_post_flag[0, 0] = 1.0

        sequence_store[uid] = {
            "post_emotion":       post_emotion_vector,
            "post_sbert":         post_sbert_vector,
            "emotion":            comment_emotion_sequence,
            "sbert":              comment_sbert_sequence,
            "emotion_with_post":  emotion_with_post,
            "sbert_with_post":    sbert_with_post,
            "is_post_flag":       is_post_flag,
            "comment_ids":        context_comments["Comment_ID"].tolist(),
        }

        context_emotion_including_post = np.vstack(
            [post_emotion_vector[None, :], comment_emotion_sequence]).mean(axis=0).tolist()
        context_sbert_including_post = np.vstack(
            [post_sbert_vector[None, :], comment_sbert_sequence]).mean(axis=0).tolist()

        thread_rows.append({
            "UID":       uid,
            "Post_ID":   thread["Post_ID"].iloc[0],
            "forum":     uid_to_forum.get(uid),
            "split":     uid_to_split.get(uid),
            "n_comments":                number_of_comments,
            "n_context_comments":        cutoff_index,
            "context_comment_ids":       json.dumps(context_comments["Comment_ID"].tolist()),
            # pre-cutoff context means (comment-only and post-included)
            "context_emotion_mean":            json.dumps(stack_mean(context_comments["Emotion_Vector"])),
            "context_sbert_mean":              json.dumps(stack_mean(context_comments["SBERT_Vector"])),
            "context_emotion_mean_with_post":  json.dumps(context_emotion_including_post),
            "context_sbert_mean_with_post":    json.dumps(context_sbert_including_post),
            # post as standalone input feature
            "post_emotion":   json.dumps(post_emotion_vector.tolist()),
            "post_sbert":     json.dumps(post_sbert_vector.tolist()),
            # targets (all post-cutoff comments, and poster-only)
            "target_emotion_mean":         json.dumps(stack_mean(target_comments["Emotion_Vector"])),
            "target_emotion_mean_poster":  (json.dumps(stack_mean(poster_target_comments["Emotion_Vector"]))
                                            if len(poster_target_comments) else None),
        })

    cutoff_frame = pd.DataFrame(thread_rows)
    csv_path      = feature_directory + f"cutoff_{cutoff_percentage}_targets.csv"
    sequence_path = feature_directory + f"cutoff_{cutoff_percentage}_sequences.pkl"

    cutoff_frame.to_csv(csv_path, index=False)
    with open(sequence_path, "wb") as file_handle:
        pickle.dump(sequence_store, file_handle, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"cutoff {cutoff_percentage}%: {len(cutoff_frame)} threads "
          f"| {cutoff_frame['split'].value_counts().to_dict()}")
    print(f"  csv  -> {csv_path}")
    print(f"  seqs -> {sequence_path}")

In [ ]:
for cutoff in [25, 50, 75]:
    df = pd.read_csv(feature_directory + f"cutoff_{cutoff}_targets.csv")
    tgt = df["target_emotion_mean"].apply(lambda v: np.array(json.loads(v)))
    n_nan = tgt.apply(lambda v: not np.all(np.isfinite(v))).sum()
    print(f"cutoff {cutoff}%: {len(df)} threads, {n_nan} NaN targets")